In [140]:
import pandas as pd
import numpy as np
import math

# google_trends = pd.read_csv("data/gold_google_trends_daily.csv")
data = pd.read_csv("files/processed_data.csv")
match_context = pd.read_csv("data/gold_match_context.csv")
# match_goals = pd.read_csv("data/gold_match_goals.csv")
match_tickets = pd.read_csv("data/gold_match_tickets.csv")
matches = pd.read_csv("data/gold_match.csv")
match_articles = pd.read_csv("data/gold_belga_press_articles.csv", on_bad_lines="skip")

In [141]:
article_count = (match_articles
                 .groupby("match_id")["match_id"]
                 .value_counts()
                 .to_frame()
                 .reset_index()
                 .rename(columns={"count": "article_count"}))

In [142]:
tickets_sold_goals = match_tickets[["tickets_sold_total", "match_id", "seasonpass_holders"]]
matchdays = matches[["matchday", "match_id"]]

data = pd.merge(data, tickets_sold_goals, on="match_id")
data = pd.merge(data, matchdays, on="match_id")
data["away_team_code"] = data["away_team_code"].astype("category")


In [143]:
data["date"] = pd.to_datetime(data["date"])
data["month"] = data['date'].dt.month
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
data = data.drop(["date", "month"], axis=1)

In [144]:
data["kickoff_time_local"] = pd.to_datetime(data["kickoff_time_local"], format='%H:%M:%S')
data["kickoff_time_local"] = pd.to_datetime(data["kickoff_time_local"], format='%H:%M:%S').dt.hour
data["is_18_hours"] = data["kickoff_time_local"] == 18
data = data.drop("kickoff_time_local", axis=1)

In [145]:
data["is_sunday"] = data["weekday"] == 6
data["is_fri_hol"] = data["weekday"].isin([4, 5, 6])

In [146]:
n = 15
train_data = data[:-n]
test_data = data.tail(n)

In [147]:
avg_tickets_scanned = train_data.groupby("away_team_code")["tickets_scanned"].mean().to_frame().rename(columns={"tickets_scanned": "avg_tickets_scanned"})
match_context_part = match_context[["academic_week", "has_promotion", "match_id"]]

In [148]:
train_data = pd.merge(train_data, match_context_part, on="match_id")
train_data = pd.merge(train_data, article_count, on="match_id")
train_data = pd.merge(train_data, avg_tickets_scanned, on="away_team_code")

test_data = pd.merge(test_data, match_context_part, on="match_id")
test_data = pd.merge(test_data, article_count, on="match_id")
test_data = pd.merge(test_data, avg_tickets_scanned, on="away_team_code")

In [149]:
train_data = train_data.drop("match_id", axis=1)
test_data = test_data.drop("match_id", axis=1)

In [150]:
test_data
# train_data

,away_team_code,last_result_vs_opponent,tickets_scanned,weekday,ohl_interest,tickets_sold_total,seasonpass_holders,matchday,month_cos,is_18_hours,is_sunday,is_fri_hol,academic_week,has_promotion,article_count,avg_tickets_scanned
0,WES,0,5025.0,4,21.09,5367,4546,9.0,-8.660254e-01,False,False,True,37,False,26,6952.250000
1,CHA,-1,4318.0,6,36.78,6066,4235,1.0,-8.660254e-01,False,True,True,48,True,25,5904.500000
2,GNK,-2,6360.0,4,78.25,7749,4235,4.0,-5.000000e-01,False,False,True,50,True,120,6889.000000
3,STA,0,6676.0,6,39.91,7502,4235,6.0,-5.000000e-01,False,True,True,53,True,33,7665.000000
4,AND,0,6977.0,4,68.52,8111,4235,9.0,-1.836970e-16,False,False,True,4,True,109,10778.666667
5,CLU,-1,7992.0,5,100.00,8549,4235,11.0,5.000000e-01,True,False,True,7,True,72,9490.333333
6,GNT,0,6373.0,6,66.67,7347,4235,13.0,8.660254e-01,False,True,True,9,True,195,6812.500000
7,STV,1,5661.0,6,47.22,7078,4235,15.0,8.660254e-01,False,True,True,12,True,11,6555.500000
8,ZWA,-2,4494.0,6,39.81,6278,4235,17.0,1.000000e+00,False,True,True,14,True,30,7785.000000
9,CER,1,5812.0,6,34.16,7473,4235,19.0,1.000000e+00,False,True,True,16,True,38,6095.666667


In [151]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error


X_train = train_data.drop("tickets_scanned", axis=1)
y_train = train_data["tickets_scanned"]

X_test = test_data.drop("tickets_scanned", axis=1)
y_test = test_data["tickets_scanned"]

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    enable_categorical=True
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)

MAE: 1338.917933872768


In [152]:
preds = pd.DataFrame(preds, columns=["predictions"])

In [153]:
tickets_scanned = test_data["tickets_scanned"]
comparing = pd.concat([tickets_scanned, preds], axis=1)

In [154]:
# comparing["diff"]
comparing["diff"] = comparing["predictions"] - comparing["tickets_scanned"]

In [155]:
comparing

,tickets_scanned,predictions,diff
0,5025.0,5609.976562,584.976562
1,4318.0,4733.690918,415.690918
2,6360.0,6907.341797,547.341797
3,6676.0,8189.594238,1513.594238
4,6977.0,9979.786133,3002.786133
5,7992.0,9513.448242,1521.448242
6,6373.0,7629.561523,1256.561523
7,5661.0,7034.052734,1373.052734
8,4494.0,7368.362305,2874.362305
9,5812.0,6468.737305,656.737305
